In [2]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The agentic loop is literally a **“while True”** loop that keeps talking to the LLM until the model tells it “I’m finished”.  
Here’s the step‑by‑step flow from the code in *14‑agentic‑loop.md*:

1. **Start with a message history** – the `messages` list contains the developer prompt, the user’s question, and any past tool calls or model replies.

2. **Send the whole history to the LLM**  
   ```python
   response = openai_client.responses.create(
       model=model,
       input=messages,
       tools=[search_tool]
   )
   ```

3. **Append the raw model output to the history**  
   `messages.extend(response.output)`

4. **Process each item in the model’s output**  
   * If it’s a **function_call**:  
     * Print/log the call.  
     * Convert the JSON arguments to a Python dict, run the actual `search()` function, serialize the result, and append a `function_call_output` back into the history.  
     * Set `has_function_calls = True`.
   * If it’s a **message**:  
     * Print the ass

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

# creates the SDK's central configuration object. It owns the span processors and decides how spans are built
provider = TracerProvider() 
# wires a processor that forwards every finished span to the console exporter, one at a time. 
# "Simple" means synchronous and immediate - good for development.
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter()) 
)

# provider.add_span_processor(
#     SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
# )

# registers the provider globally, so every call to trace.get_tracer(...) returns a tracer backed by it
trace.set_tracer_provider(provider)

# returns a Tracer we use to create spans. 
# The string is just a label for the instrumentation scope - it identifies which part of the code produced the spans.
tracer = trace.get_tracer("llm-zoomcamp")

Overriding of current TracerProvider is not allowed


In [4]:
with tracer.start_as_current_span("my_operation") as span:
    # your code here
    span.set_attribute("my_key", "my_value")

{
    "name": "my_operation",
    "context": {
        "trace_id": "0x033ca3fe94689ea00404dfa4bc536161",
        "span_id": "0xfe73186035bddd48",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-08-04T07:26:32.610667Z",
    "end_time": "2026-08-04T07:26:32.610697Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "my_key": "my_value"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "362d6580-a1f1-4809-8ed1-d739f4103832",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


In [5]:
from rag_helper import RAGBase


class RAGTraced(RAGBase):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.tracer = trace.get_tracer(__name__)

    def rag(self, query: str) -> str:
        with self.tracer.start_as_current_span("rag_operation") as span:
            span.set_attribute("query", query)
            answer = super().rag(query)
            span.set_attribute("answer", answer)
            return answer
    def llm(self, query: str) -> str:
        with self.tracer.start_as_current_span("llm_operation") as span:
            span.set_attribute("query", query)
            response = super().llm(query)
            span.set_attribute("answer", response.output_text)
            span.set_attribute("input_tokens", response.usage.input_tokens)
            span.set_attribute("output_tokens", response.usage.output_tokens)
            return response
    def search(self, query, num_results=5):
        with self.tracer.start_as_current_span("search_operation") as span:
            span.set_attribute("query", query)
            span.set_attribute("num_results", num_results)
            return super().search(query, num_results)
        

In [6]:
import os

from gitsource import GithubRepositoryDataReader
from openai import OpenAI

from minsearch import Index
from gitsource import GithubRepositoryDataReader

from dotenv import load_dotenv
load_dotenv()


index = Index(text_fields=["content"], keyword_fields=["filename"])

COMMIT = "8c1834d"

# --- Load the course lessons (same as HW1, HW2, HW4) ---
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)


# client = OpenAI()
client = OpenAI(
        api_key=os.getenv("GROQ_API_KEY"),
        base_url="https://api.groq.com/openai/v1"
    )
llm_model = "openai/gpt-oss-20b"
rag_traced = RAGTraced(
    index=index, 
    llm_client=client,
    model=llm_model)



In [29]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)
print(answer)

{
    "name": "search_operation",
    "context": {
        "trace_id": "0x8bbcb3bf8adc5fc31e6666bab47025f2",
        "span_id": "0xc62e4a73082149b1",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x6a50dfa41cbe6d42",
    "start_time": "2026-08-04T08:56:22.919810Z",
    "end_time": "2026-08-04T08:56:22.922086Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "362d6580-a1f1-4809-8ed1-d739f4103832",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_operation",
    "context": {
        "trace_id": 

Q1 : The trace produced 3 Spans.

Q2. Capturing metrics as span attributes => 7174 (~7000)



In [26]:
from datetime import datetime

start = datetime.fromisoformat("2026-08-04T07:44:55.367223Z".replace("Z", "+00:00"))
end = datetime.fromisoformat("2026-08-04T07:44:56.760182Z".replace("Z", "+00:00"))

elapsed_ms = (end - start).total_seconds() * 1000

print(elapsed_ms)

1392.959


In [9]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [10]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

# creates the SDK's central configuration object. It owns the span processors and decides how spans are built
provider = TracerProvider() 
# wires a processor that forwards every finished span to the console exporter, one at a time. 
# "Simple" means synchronous and immediate - good for development.


provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

# registers the provider globally, so every call to trace.get_tracer(...) returns a tracer backed by it
trace.set_tracer_provider(provider)

# returns a Tracer we use to create spans. 
# The string is just a label for the instrumentation scope - it identifies which part of the code produced the spans.
tracer = trace.get_tracer("llm-zoomcamp")

In [19]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)
print(answer)

**Short answer**

The agentic loop is just a `while True` that

1. sends the current conversation to the model,  
2. looks at the response,  
3. if the response contains a **function‑call** it runs that tool,  
4. adds the tool’s output back to the conversation, and  
5. repeats until the response contains **no** function calls.  

When a turn comes back without a `function_call`, the loop stops – that means the LLM has decided it can answer the user directly.

---

### How it works step by step (from the code)

```python
it = 1
while True:
    # 1️⃣  Ask the model
    response = openai_client.responses.create(
        model=model,
        input=messages,
        tools=[search_tool]
    )

    # 2️⃣  Append everything the model returned to the history
    messages.extend(response.output)

    # 3️⃣  Scan the turn for function calls
    has_function_calls = False
    for item in response.output:
        if item.type == "function_call":
            # 3a.  Run the tool (search) and get th

In [18]:
import sqlite3

conn = sqlite3.connect("traces.db")
cursor = conn.cursor()

cursor.execute("SELECT * FROM spans")

rows = cursor.fetchall()
for row in rows:
    operation, start, end, input_tokens, output_tokens, cost = row
    print(f"Operation: {operation}, Elapsed Time ; {end-start}, Input Tokens: {input_tokens}, Output Tokens: {output_tokens}, Cost: {cost}")

conn.close()

Operation: search_operation, Elapsed Time ; 6085165, Input Tokens: None, Output Tokens: None, Cost: None
Operation: llm_operation, Elapsed Time ; 24164988531, Input Tokens: 7174, Output Tokens: 571, Cost: None
Operation: rag_operation, Elapsed Time ; 24188914031, Input Tokens: None, Output Tokens: None, Cost: None
Operation: search_operation, Elapsed Time ; 5190602, Input Tokens: None, Output Tokens: None, Cost: None
Operation: llm_operation, Elapsed Time ; 2077372879, Input Tokens: 7174, Output Tokens: 904, Cost: None
Operation: rag_operation, Elapsed Time ; 2100220252, Input Tokens: None, Output Tokens: None, Cost: None
Operation: search_operation, Elapsed Time ; 2774195, Input Tokens: None, Output Tokens: None, Cost: None
Operation: llm_operation, Elapsed Time ; 51883704750, Input Tokens: 7174, Output Tokens: 447, Cost: None
Operation: rag_operation, Elapsed Time ; 51899441917, Input Tokens: None, Output Tokens: None, Cost: None


Q5 : LLM span takes most total time excluding rag

In [ ]:
Load the SQLite data with pandas. One thing a dashboard can tell you is how stable your system is. If the same query always produces the same number of input tokens, the context your RAG retrieves is consistent. If it varies a lot, something in the search may be unstable.

Run the same query from Q1 three more times (so you have 4 RAG calls total in the database). Then compute the input tokens for each llm span.

How much do the input tokens vary across these 4 runs?

They're identical
Within 10% of each other
Within 50% of each other
They vary more than 50%

In [22]:
import pandas as pd

conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("SELECT * FROM spans", conn)
df.head()

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search_operation,1785834059333277130,1785834059339362295,NaN,NaN,None
1,llm_operation,1785834059347797072,1785834083512785603,7174.0,571.0,None
2,rag_operation,1785834059333139547,1785834083522053578,NaN,NaN,None
3,search_operation,1785834366314514385,1785834366319704987,NaN,NaN,None
4,llm_operation,1785834366329217218,1785834368406590097,7174.0,904.0,None


In [23]:
df[df['name'] == 'llm_operation']['input_tokens'].describe()

count       4.0
mean     7174.0
std         0.0
min      7174.0
25%      7174.0
50%      7174.0
75%      7174.0
max      7174.0
Name: input_tokens, dtype: float64